In [1]:
import numpy as np
import pandas as pd
import requests
import json
import dotenv
import os
import yaml
import llm

from lxml import etree
from lxml import html

### Getting Current and Historical Committee Assignment from unitedstates/congress-legislators

In [2]:
# Current committees
url = "https://raw.githubusercontent.com/unitedstates/congress-legislators/main/committees-current.yaml"
response = requests.get(url)
current = yaml.safe_load(response.text)

In [3]:
pd.DataFrame(current).head()

,type,name,url,minority_url,thomas_id,house_committee_id,subcommittees,address,phone,rss_url,jurisdiction,youtube_id,jurisdiction_source,minority_rss_url,senate_committee_id,wikipedia
0,house,House Committee on Agriculture,https://agriculture.house.gov/,https://republicans-agriculture.house.gov,HSAG,AG,"[{'name': 'Forestry and Horticulture', 'thomas...","1301 LHOB; Washington, DC 20515-6001",(202) 225-2171,https://agriculture.house.gov/Rss.aspx?GroupID=1,The House Committee on Agriculture has legisla...,UCOWh2WJxPywHIaccDWb8Mvg,NaN,NaN,NaN,NaN
1,house,House Committee on Appropriations,https://appropriations.house.gov/,https://republicans-appropriations.house.gov/,HSAP,AP,"[{'name': 'Agriculture, Rural Development, Foo...","H307 CAPITOL; Washington, DC 20515-6015",(202) 225-2771,NaN,The House Committee on Appropriations is respo...,UCMaSlF09S0fpoRshS2t_7XA,https://appropriations.house.gov/about/,NaN,NaN,NaN
2,house,House Committee on Armed Services,https://armedservices.house.gov/,https://republicans-armedservices.house.gov/,HSAS,AS,"[{'name': 'Tactical Air and Land Forces', 'tho...","2216 RHOB; Washington, DC 20515-6035",(202) 225-4151,NaN,The House Committee on Armed Services has legi...,UCD506yORW2voSanqEgLOUIQ,https://armedservices.house.gov/about,NaN,NaN,NaN
3,house,House Committee on Financial Services,https://financialservices.house.gov/,https://republicans-financialservices.house.gov/,HSBA,BA,"[{'name': 'Capital Markets', 'thomas_id': '16'...","2129 RHOB; Washington, DC 20515-6050",(202) 225-7502,NaN,The House Financial Services Committee has jur...,UCiGw0gRK-daU7Xv4oDMr9Hg,https://financialservices.house.gov/about/,NaN,NaN,NaN
4,house,House Committee on the Budget,https://budget.house.gov/,https://democrats-budget.house.gov/,HSBU,BU,NaN,"204 CHOB; Washington, DC 20515-6065",(202) 226-7270,https://budget.house.gov/news/rss.aspx,The House Committee on the Budget is responsib...,UCwzia2rpHJkowAXK-IF9E0w,https://budget.house.gov/about/,https://democrats-budget.house.gov/rss.xml,NaN,NaN


In [7]:
# Current membership
url = "https://raw.githubusercontent.com/unitedstates/congress-legislators/main/committee-membership-current.yaml"
response = requests.get(url)
current_membership = yaml.safe_load(response.text)

In [8]:
pd.DataFrame(current_membership).head()

ValueError: All arrays must be of the same length

In [4]:
# Historical committees (since 1973; 93rd Congress)
url = "https://raw.githubusercontent.com/unitedstates/congress-legislators/main/committees-historical.yaml"
response = requests.get(url)
historical = yaml.safe_load(response.text)

In [5]:
pd.DataFrame(historical)

,type,name,thomas_id,house_committee_id,congresses,names,subcommittees,senate_committee_id
0,house,House Select Subcommittee on the Coronavirus P...,HSVC,VC,"[117, 118]",NaN,NaN,NaN
1,house,House Select Subcommittee on the Weaponization...,HSFD,FD,[118],NaN,NaN,NaN
2,house,House Task Force on the Attempted Assassinatio...,HSZT,ZT,[118],NaN,NaN,NaN
3,house,House Committee on Energy (Ad Hoc),HHAH,AH,[95],{95: 'Energy (Ad Hoc)'},NaN,NaN
4,house,House Committee on Committees (Select),HLCQ,CQ,"[93, 96]","{93: 'Committees (Select)', 96: 'Committees (S...",NaN,NaN
...,...,...,...,...,...,...,...,...
65,house,House Select Committee on Economic Disparity a...,HSEF,EF,[117],{117: 'Economic Disparity and Fairness in Grow...,NaN,NaN
66,house,House Select Committee on the Modernization of...,HSMH,MH,[117],{117: 'Modernization of Congress'},NaN,NaN
67,house,House Select Committee to Investigate the Janu...,HSIJ,IJ,[117],{117: 'Investigate the January 6th Attack on t...,NaN,NaN
68,senate,Commission on Security and Cooperation in Euro...,JCSE,NaN,"[112, 113]",{112: 'Commission on Security and Cooperation ...,NaN,NaN


In [6]:
historical

[{'type': 'house',
  'name': 'House Select Subcommittee on the Coronavirus Pandemic',
  'thomas_id': 'HSVC',
  'house_committee_id': 'VC',
  'congresses': [117, 118]},
 {'type': 'house',
  'name': 'House Select Subcommittee on the Weaponization of the Federal Government',
  'thomas_id': 'HSFD',
  'house_committee_id': 'FD',
  'congresses': [118]},
 {'type': 'house',
  'name': 'House Task Force on the Attempted Assassination of Donald J. Trump',
  'thomas_id': 'HSZT',
  'house_committee_id': 'ZT',
  'congresses': [118]},
 {'type': 'house',
  'name': 'House Committee on Energy (Ad Hoc)',
  'thomas_id': 'HHAH',
  'house_committee_id': 'AH',
  'names': {95: 'Energy (Ad Hoc)'},
  'congresses': [95]},
 {'type': 'house',
  'name': 'House Committee on Committees (Select)',
  'thomas_id': 'HLCQ',
  'house_committee_id': 'CQ',
  'names': {93: 'Committees (Select)', 96: 'Committees (Select)'},
  'congresses': [93, 96]},
 {'type': 'house',
  'name': 'House Committee on Outer Continental Shelf (Sel

In [14]:
### Data from Stewart's code
stewart_house = pd.read_excel("../data/Stewart_House_103-117.xls")
stewart_senate = pd.read_excel("../data/Stewart_Senate_103-117.xlsx")

pd.DataFrame(stewart_house).head()
pd.DataFrame(stewart_senate).head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/Stewart_House_103-117.xls'

### Historical Committee MemberData from GovTrack

In [6]:
# Funcion to parse a GovTrack historicla cmt membership XML
def parse_cmt_xml(congress_num):    
    url = f"https://raw.githubusercontent.com/govtrack/historical-committee-membership/main/{congress_num}.xml"
    response = requests.get(url)
    response.raise_for_status()
    root = html.fromstring(response.content)

    rows = []
    for committee in root.findall("committee"):
        c_type = committee.get("type")
        c_code = committee.get("code")
        c_name = committee.get("displayname")

        # Direct committee members (no subcommittee)
        for member in committee.findall("member"):
            rows.append({
                "congress": congress_num,
                "type": c_type,
                "committee_code": c_code,
                "committee_name": c_name,
                "subcommittee_code": None,
                "subcommittee_name": None,
                "member_id": member.get("id"),
                "role": member.get("role"),
            })

        # Subcommittee members
        for sub in committee.findall("subcommittee"):
            s_code = sub.get("code")
            s_name = sub.get("displayname")
            for member in sub.findall("member"):
                rows.append({
                    "congress": congress_num,
                    "type": c_type,
                    "committee_code": c_code,
                    "committee_name": c_name,
                    "subcommittee_code": s_code,
                    "subcommittee_name": s_name,
                    "member_id": member.get("id"),
                    "role": member.get("role"),
                })

    return pd.DataFrame(rows)


# Parse the 109th Congress as an example
df_109 = parse_cmt_xml(109)


In [7]:
df_109.head(10)

,congress,type,committee_code,committee_name,subcommittee_code,subcommittee_name,member_id,role
0,109,house,HSAG,House Committee on Agriculture,NaN,NaN,400154,Chairman
1,109,house,HSAG,House Committee on Agriculture,NaN,NaN,400322,Vice Chairman
2,109,house,HSAG,House Committee on Agriculture,NaN,NaN,400127,NaN
3,109,house,HSAG,House Committee on Agriculture,NaN,NaN,400247,NaN
4,109,house,HSAG,House Committee on Agriculture,NaN,NaN,400284,NaN
5,109,house,HSAG,House Committee on Agriculture,NaN,NaN,400202,NaN
6,109,house,HSAG,House Committee on Agriculture,NaN,NaN,400164,NaN
7,109,house,HSAG,House Committee on Agriculture,NaN,NaN,400172,NaN
8,109,house,HSAG,House Committee on Agriculture,NaN,NaN,400207,NaN
9,109,house,HSAG,House Committee on Agriculture,NaN,NaN,400303,NaN
